In [20]:
import os
from os.path import isfile, join
import glob
from pydicom import dcmread
from pydicom.multival import MultiValue
import re
import numpy as np
from utils import remove_overlap
import warnings
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

"""
Discard reasons:
Left
ANON1832: No dose, rtstruct, ct, or mask
ANON1845, ANON1861, ANON0090, ANON0091, ANON1792, ANON0132: Missing critical structures (PTV, heart, lungs or contra breast)
"""

def downsample_dicom_folder(dataset):
    if dataset == 'L':
        DISCARD_LIST = ['ANON1845', 'ANON1861', 'ANON0090', 'ANON0091', 'ANON1792', 'ANON0132']
        BEAMS = '.+2\.beams'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_L/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_L_ds/"
    elif dataset == 'R':
        DISCARD_LIST = ['']
        BEAMS = '.+2\.beams'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_R/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_R_ds/"
    elif dataset == 'LAX':
        DISCARD_LIST = ['ANON0941']
        BEAMS = '.+1\.beam'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_LAX/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_LAX_ds/"
    elif dataset == 'RAX':
        DISCARD_LIST = []
        BEAMS = '.+1\.beam'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_RAX/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/uudet/Breast_RAX_ds/"
        
    all_items = glob.glob(SOURCE_PATH, recursive=True)
    study_folders = [s for s in all_items if 'Anon' in s]
    r = re.compile('ANON\d{4}')
    unique_subjects = [r.search(s)[0] for s in study_folders if r.search(s)]
    unique_subjects = np.unique(unique_subjects)
    
    for i, subject in enumerate(unique_subjects):
        if not subject in DISCARD_LIST: # These are 50Gy or SIB, thus discarding
            print(f"{i}: {subject}")
            r = re.compile(f"{subject}(.+RTst|{BEAMS})")
            subject_folders = [s for s in study_folders if r.search(s)]
            # There should be CT, mask "CT", dose, and RTstruct for each subject.
            if len(subject_folders) == 0:
                pass
            else:
                if len(subject_folders) != 4:
                    warnings.warn("Subject " + subject + " has != 5 folders. Ensure manually that ct, dose, and mask are present. Patient not processed.")
                    continue

                r_ct = re.compile('(?!.*Mask.*Mask).+CT.+CT.+CT')
                #r_ct = re.compile('(?!.*Mask.*Mask).+CT.+CT.+DVH.+CT')
                r_mask = re.compile('.+Mask.+Mask')
                r_dose = re.compile('.+Dose')

                ct_path = [s for s in subject_folders if r_ct.search(s)]
                mask_path = [s for s in subject_folders if r_mask.search(s)]
                dose_path = [s for s in subject_folders if r_dose.search(s)]
                
                dcm_path_ct = os.listdir(ct_path[0])
                dcmfile_path_dose = dose_path[0] + os.listdir(dose_path[0])[0]
                dcm_path_mask = os.listdir(mask_path[0])
                
                ### Ensin tehdään potilaskansio
                try:
                    os.mkdir(DESTINATION_PATH + subject)
                except FileExistsError:
                    pass

                
                ### Tallennetaan dose ekana. Tämä hieman erilainen kuin muut, koska kaikki leikkeet samassa filessä.
                dcm_dose = dcmread(dcmfile_path_dose)
                # Downsamplaus - harkitse filtteröintiä jos tätä suurempi kerroin
                dcm_dose_downsampled = zoom(dcm_dose.pixel_array, zoom=(1, 0.5, 0.5), order=1)

                #if dataset == 'RAX':
                #    dose_copy = dcm_dose_downsampled.copy()
                
                # Päivitetään DICOMin muotokentät
                _, dcm_dose.Rows, dcm_dose.Columns = dcm_dose_downsampled.shape
                
                # Päivitettään myös pixel spacing, jotta pikselin koko edelleen oikein. Tuo spacing on decimal string (ds) ja sen tyyppi on
                # pydicomin MultiValue, siksi muutettu hieman hassusti.
                new_spacing = [float(x)*2 for x in list(dcm_dose.PixelSpacing)]
                dcm_dose.PixelSpacing = MultiValue(float, new_spacing)

                try:
                    os.mkdir(DESTINATION_PATH + subject + "/dose/")
                except FileExistsError:
                    pass
                
                
                ### Sitten CT
                try:
                    os.mkdir(DESTINATION_PATH + subject + "/ct/")
                except FileExistsError:
                    pass
                
                # Tässä joudutaan iteroimaan läpi koko kansio ja avataan ja tallennetaan yksittäiset leikkeet yksitellen.
                for dcmfile_ct in dcm_path_ct:
                    
                        
                    dcm_ct = dcmread(ct_path[0] + dcmfile_ct)
                    dcm_ct_downsampled = zoom(dcm_ct.pixel_array, zoom=(0.5, 0.5), order=1)
                    dcm_ct.PixelData = dcm_ct_downsampled.tobytes()


                    dcm_ct.Rows, dcm_ct.Columns = dcm_ct_downsampled.shape
                    new_spacing = [float(x)*2 for x in list(dcm_ct.PixelSpacing)]
                    dcm_ct.PixelSpacing = MultiValue(float, new_spacing)
                    dcm_ct.save_as(DESTINATION_PATH + subject + "/ct/" + dcmfile_ct)
                
                ### Maskit täysin samalla tavalla kuin CT
                try:
                    os.mkdir(DESTINATION_PATH + subject + "/mask/")
                except FileExistsError:
                    pass
                
                for dcmfile_mask in dcm_path_mask:
                    dcm_mask = dcmread(mask_path[0] + dcmfile_mask)
                    dcm_mask_downsampled = zoom(dcm_mask.pixel_array, zoom=(0.5, 0.5), order=0)
                    dcm_mask.PixelData = dcm_mask_downsampled.tobytes()

                    # Instance number is basically slice number, reversed compared to index of dose array
                    instance_number = dcm_mask.InstanceNumber
                    instance_number_reversed = np.shape(dcm_dose_downsampled)[0] - instance_number
                    
                    dcm_mask.Rows, dcm_mask.Columns = dcm_mask_downsampled.shape
                    mask_min = np.min(dcm_mask_downsampled)
                    
                    dcm_dose_downsampled[instance_number_reversed, :, :] = np.multiply(dcm_dose_downsampled[instance_number_reversed, :, :], np.isin(dcm_mask_downsampled, mask_min, invert = True)) #1023 equals to -1

                    new_spacing = [float(x)*2 for x in list(dcm_mask.PixelSpacing)]
                    dcm_mask.PixelSpacing = MultiValue(float, new_spacing)
                    dcm_mask.save_as(DESTINATION_PATH + subject + "/mask/" + dcmfile_mask)
    
                # Haetaan tallennetaan pienennetty data takaisin DICOMiin
                dcm_dose.PixelData = dcm_dose_downsampled.tobytes()
                dcm_dose.save_as(DESTINATION_PATH + subject + "/dose/" + os.listdir(dose_path[0])[0])


print("PROCESSING...")
downsample_dicom_folder('LAX')
print("DONE")

# n_L = 279

PROCESSING...
0: ANON0100
1: ANON0271
2: ANON0842
3: ANON0844
4: ANON0847
5: ANON0848
6: ANON0849
7: ANON0850
8: ANON0853
9: ANON0857
10: ANON0859
11: ANON0863
12: ANON0866
13: ANON0868
14: ANON0869
15: ANON0870
16: ANON0871
17: ANON0872
18: ANON0873
19: ANON0874
20: ANON0878
21: ANON0879
22: ANON0883
23: ANON0887
24: ANON0888


/tmp/ipykernel_2824301/1850616400.py:58: UserWarning: Subject ANON0887 has != 5 folders. Ensure manually that ct, dose, and mask are present. Patient not processed.
  warnings.warn("Subject " + subject + " has != 5 folders. Ensure manually that ct, dose, and mask are present. Patient not processed.")


25: ANON0894
26: ANON0896
27: ANON0900
28: ANON0903
29: ANON0904
30: ANON0908
31: ANON0913
32: ANON0915
33: ANON0916
34: ANON0919
35: ANON0920
36: ANON0922
37: ANON0926
38: ANON0929
39: ANON0932
40: ANON0934
41: ANON0935
42: ANON0936
43: ANON0937
44: ANON0939
45: ANON0944
46: ANON0946
47: ANON0949
48: ANON0950
49: ANON0953
50: ANON0957
51: ANON0958
52: ANON0959
53: ANON0961
54: ANON0962
55: ANON0963
56: ANON0966
57: ANON0967
58: ANON0968
59: ANON0970
60: ANON0972
61: ANON0973
62: ANON0974
63: ANON0977
64: ANON0978
65: ANON0980
66: ANON0981
67: ANON0983
68: ANON0986
69: ANON0988
70: ANON0991
71: ANON0992
72: ANON0993
73: ANON0994
74: ANON0995
75: ANON0996
76: ANON0997
77: ANON0998
78: ANON1000
79: ANON1003
80: ANON1008
81: ANON1009
82: ANON1010
83: ANON1011
84: ANON1015
85: ANON1016
86: ANON1018
87: ANON1021
88: ANON1022
89: ANON1023
90: ANON1025
91: ANON1163
92: ANON1172
93: ANON1177
DONE


In [15]:
subject_folders

NameError: name 'subject_folders' is not defined

In [19]:
np.any(np.isin([1, 2, 3, 4, 5], [1, 2]))

True

In [5]:
dcm_mask

NameError: name 'dcm_mask' is not defined

NameError: name 'training_set' is not defined